# MRI Severity Classifier (OASIS brain scans, 3 levels)

**Data:** Kaggle `kanaadlimaye/alzheimers-classification-dataset` (OASIS MRI slices, 224x224).
**Task:** classify a brain MRI slice as None / Very mild / Mild-to-moderate.
**Method:** transfer learning with InceptionResNetV2 (ImageNet weights), then fine-tuning.

Run on a GPU: Runtime > Change runtime type > T4 GPU.

Things this notebook handles on purpose:
- Moderate (MoD) has only 88 / 15 / 5 images in train / valid / test, so MD and MoD are merged into one class.
- The train folder holds each source image twice (flipped/rotated copies made by the dataset authors). We keep them as is: no image appears in more than one split, so this is not leakage.
- Class weights, because the classes are imbalanced.
- Per-class recall and a confusion matrix on the untouched test set, not just accuracy.
- Grad-CAM heatmaps to show where the model looked.
- Limitation: the dataset has no patient ID, so we cannot rule out slices of one person appearing in several splits.

## 1. Setup and data download

In [ ]:
!pip -q install kaggle
import os
from google.colab import userdata
# Each person adds their own Kaggle token in Colab: left sidebar > Secrets > name KAGGLE_API_TOKEN
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
!kaggle datasets download -d kanaadlimaye/alzheimers-classification-dataset -p /content/data --unzip
ROOT = "/content/data/Alzheimers_Detection_dataset"
print(os.listdir(ROOT))

## 2. Build the 3-class tables (merge MD+MoD, drop duplicate source images)

In [ ]:
import re, numpy as np, pandas as pd, tensorflow as tf
SEED = 42
tf.keras.utils.set_random_seed(SEED)

CLASSES = ["None (ND)", "Very mild (VMD)", "Mild-moderate (MD+MoD)"]
def load(split):
    df = pd.read_csv(f"{ROOT}/CSV_datafiles/_{split}_classes.csv")
    # Label cleaning: 21 CSV rows are all zeros and 7 disagree with the filename, so the label is taken from the filename prefix (ND__, VMD__, MD__, MoD__)
    lab = df["filename"].str.split("__").str[0].map({"ND":0,"VMD":1,"MD":2,"MoD":2})
    out = pd.DataFrame({"path": f"{ROOT}/{split}/" + df["filename"], "y": lab})
    out["src"] = df["filename"].str.replace(r"_jpg\.rf\..*", "", regex=True)
    return out

train, valid, test = load("train"), load("valid"), load("test")
# Train keeps both augmented copies of each source image; the splits share no images, so there is no leakage across splits.
for n, d in [("train",train),("valid",valid),("test",test)]:
    print(n, len(d), d.y.value_counts().sort_index().to_dict())

## 3. Input pipeline

In [ ]:
from tensorflow.keras.applications.inception_resnet_v2 import preprocess_input
BATCH = 32
def make_ds(df, training):
    def read(p, y):
        img = tf.io.decode_jpeg(tf.io.read_file(p), channels=3)
        img = tf.image.resize(img, (224,224))
        return preprocess_input(img), y
    ds = tf.data.Dataset.from_tensor_slices((df.path.values, df.y.values.astype("int32")))
    if training: ds = ds.shuffle(len(df), seed=SEED)
    return ds.map(read, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds, valid_ds, test_ds = make_ds(train,True), make_ds(valid,False), make_ds(test,False)

# class weights: rarer class counts more
counts = np.bincount(train.y, minlength=3)
cw = {i: float(len(train)/(3*counts[i])) for i in range(3)}
print("class weights:", cw)

## 4. Model (frozen base first, then fine-tune)

In [ ]:
from tensorflow.keras import layers, Model
base = tf.keras.applications.InceptionResNetV2(include_top=False, weights="imagenet", input_shape=(224,224,3))
base.trainable = False

inp = layers.Input((224,224,3))
x = base(inp, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(3, activation="softmax")(x)
model = Model(inp, out)

cbs = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)]
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h1 = model.fit(train_ds, validation_data=valid_ds, epochs=15, class_weight=cw, callbacks=cbs)

In [ ]:
# Fine-tune: unfreeze the top of the base model with a very small learning rate
base.trainable = True
for l in base.layers[:-40]: l.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h2 = model.fit(train_ds, validation_data=valid_ds, epochs=15, class_weight=cw, callbacks=cbs)
model.save("mri_3class.keras")

## 5. Test-set evaluation (the 640 images never used for training or tuning)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt, seaborn as sns
prob = model.predict(test_ds)
pred = prob.argmax(1)
print(classification_report(test.y, pred, target_names=CLASSES, digits=3))
cm = confusion_matrix(test.y, pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Test confusion matrix"); plt.show()

## 6. Grad-CAM: where did the model look?

In [ ]:
last_conv = base.get_layer("conv_7b_ac")
grad_model = Model(base.input, last_conv.output)   # conv_7b_ac is the base output
head_layers = model.layers[model.layers.index(model.get_layer(base.name))+1:]

def gradcam(img):                          # img: preprocessed (224,224,3)
    with tf.GradientTape() as tape:
        conv = grad_model(img[None])
        tape.watch(conv)
        z = conv
        for l in head_layers: z = l(z, training=False)
        cls = tf.argmax(z[0]); score = z[:, cls]
    g = tape.gradient(score, conv)
    w = tf.reduce_mean(g, axis=(1,2))
    cam = tf.nn.relu(tf.reduce_sum(conv * w[:,None,None,:], axis=-1))[0].numpy()
    cam = cam / (cam.max() + 1e-8)
    return tf.image.resize(cam[...,None], (224,224)).numpy()[...,0], int(cls)

idx = np.random.RandomState(SEED).choice(len(test), 8, replace=False)
fig, ax = plt.subplots(2, 8, figsize=(24,6))
for j, i in enumerate(idx):
    raw = tf.image.resize(tf.io.decode_jpeg(tf.io.read_file(test.path[i]), channels=3), (224,224))
    cam, c = gradcam(preprocess_input(raw))
    ax[0,j].imshow(raw.numpy().astype("uint8")); ax[0,j].set_title(f"true: {test.y[i]}  pred: {c}"); ax[0,j].axis("off")
    ax[1,j].imshow(raw.numpy().astype("uint8")); ax[1,j].imshow(cam, cmap="jet", alpha=0.45); ax[1,j].axis("off")
plt.show()

## 7. Notes to report
- Report per-class recall from section 5. Accuracy alone hides weak classes.
- Test set has only 5 moderate images originally; after the merge the mild-moderate class has 89 test images.
- No patient ID in this dataset: results may be optimistic if one person's slices are in several splits.
- This model sees only images. It does not use test scores, age or clinical data.